In [1]:
!pip install pandas faiss-cpu sentence-transformers langchain

   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.1 MB ? eta -:--:--
    --------------------------------------- 0.3/16.1 MB ? eta -:--:--
   -- ------------------------------------- 1.0/16.1 MB 2.2 MB/s eta 0:00:07
   ----- ---------------------------------- 2.1/16.1 MB 3.0 MB/s eta 0:00:05
   --------- ------------------------------ 3.7/16.1 MB 4.1 MB/s eta 0:00:04
   ------------- -------------------------- 5.2/16.1 MB 4.8 MB/s eta 0:00:03
   ---------------- ----------------------- 6.8/16.1 MB 5.3 MB/s eta 0:00:02
   -------------------- ------------------- 8.4/16.1 MB 5.6 MB/s eta 0:00:02
   ------------------------- -------------- 10.2/16.1 MB 6.0 MB/s eta 0:00:01
   ----------------------------- ---------- 12.1/16.1 MB 6.2 MB/s eta 0:00:01
   ------------------------------- -------- 12.8/16.1 MB 6.3 MB/s eta 0:00:01
   ---------------------

In [2]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

C:\Users\M A D I N A\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("Customer Support Data Set.csv")

df.head()

,question,answer,category
0,Where is my order?,You can track your order using the tracking li...,Shipping
1,How do I return a shirt that doesn't fit?,You can return any item within 30 days in its ...,Returns
2,"My package hasn't arrived yet, what should I do?",Please check the tracking number. If it hasn't...,Shipping
3,Can I exchange a dress for a different size?,"Yes, use our exchange form online to select th...",Returns
4,Do you offer free shipping?,"Yes, orders over $50 qualify for free standard...",Shipping


In [4]:
print(df.shape)

df.info()

(500, 3)
<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   question  500 non-null    str  
 1   answer    500 non-null    str  
 2   category  500 non-null    str  
dtypes: str(3)
memory usage: 71.0 KB


In [5]:
documents = []

for _, row in df.iterrows():
    text = f"""
    Question: {row['question']}
    
    Answer: {row['answer']}
    
    Category: {row['category']}
    """
    
    documents.append(text)

print(documents[0])


    Question: Where is my order?

    Answer: You can track your order using the tracking link sent to your email.

    Category: Shipping
    


In [6]:
model = SentenceTransformer('all-MiniLM-L6-v2')

C:\Users\M A D I N A\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\M A D I N A\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1886.08

In [7]:
embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches: 100%|██████████| 16/16 [00:09<00:00,  1.73it/s]

(500, 384)


In [8]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

print("Documents stored:", index.ntotal)

Documents stored: 500


In [9]:
def retrieve(query, top_k=3):
    
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    )
    
    distances, indices = index.search(
        query_embedding,
        top_k
    )
    
    results = []
    
    for idx in indices[0]:
        results.append(documents[idx])
    
    return results

In [10]:
retrieve("How can I return my order?")

['\n    Question: How do I request a refund?\n\n    Answer: Submit a request through our returns portal with your order details.\n\n    Category: Returns\n    ',
 '\n    Question: My package was lost. What should I do?\n\n    Answer: Contact support immediately; we will issue a replacement or refund if necessary.\n\n    Category: Shipping\n    ',
 '\n    Question: My item is defective. How do I return it?\n\n    Answer: Use our returns portal to report the defect; we‚Äôll send a replacement.\n\n    Category: Returns\n    ']

In [11]:
chat_history = []

In [12]:
def chatbot(query):
    
    retrieved_docs = retrieve(query)
    
    chat_history.append(
        f"User: {query}"
    )
    
    response = retrieved_docs[0]
    
    chat_history.append(
        f"Bot: {response}"
    )
    
    return response

In [13]:
chatbot("Where is my order?")

'\n    Question: Where is my order?\n\n    Answer: You can track your order using the tracking link sent to your email.\n\n    Category: Shipping\n    '

In [14]:
chatbot("How do I return a shirt?")

"\n    Question: How do I return a shirt that doesn't fit?\n\n    Answer: You can return any item within 30 days in its original condition using our return portal.\n\n    Category: Returns\n    "

In [15]:
for item in chat_history:
    print(item)
    print()

User: Where is my order?

Bot: 
    Question: Where is my order?

    Answer: You can track your order using the tracking link sent to your email.

    Category: Shipping
    

User: How do I return a shirt?

Bot: 
    Question: How do I return a shirt that doesn't fit?

    Answer: You can return any item within 30 days in its original condition using our return portal.

    Category: Returns
    



In [ ]:
while True:
    
    query = input("You: ")
    
    if query.lower() == "exit":
        break
    
    response = chatbot(query)
    
    print("\nBot:")
    print(response)
    print()

You:  how do i return a shirt that doesnt fit?



Bot:

    Question: How do I return a shirt that doesn't fit?

    Answer: You can return any item within 30 days in its original condition using our return portal.

    Category: Returns
    



You:  where is my order ?



Bot:

    Question: Where is my order?

    Answer: You can track your order using the tracking link sent to your email.

    Category: Shipping
    

